# 🧙‍♂️ Магические методы (часть 3): менеджеры контекста, `__new__`, `__del__`

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; color: white;">
  <h2 style="color: white;">Управление жизненным циклом объектов в Python</h2>
  <p>Сегодня мы погрузимся в три важнейших магических метода, которые управляют созданием, использованием и уничтожением объектов:</p>
  <ul>
    <li><code>__enter__</code> и <code>__exit__</code> — контекстные менеджеры (блоки <code>with</code>)</li>
    <li><code>__new__</code> — настоящий конструктор (создание объекта)</li>
    <li><code>__del__</code> — финализатор (уничтожение объекта)</li>
  </ul>
  <p>Эти методы позволяют писать более безопасный, эффективный и идиоматичный код.</p>
</div>

## 🎯 Цели лекции

- Понять, зачем нужны менеджеры контекста и научиться реализовывать свои с помощью `__enter__` и `__exit__`
- Разобраться в отличии `__new__` от `__init__` и научиться применять `__new__` для решения нестандартных задач (синглтон, неизменяемые объекты)
- Осознать ограничения `__del__` и освоить правильные способы управления ресурсами
- Увидеть практические примеры использования каждого метода

## 📚 План лекции

1. **Менеджеры контекста: `__enter__` и `__exit__`**
   - Проблема управления ресурсами
   - Синтаксис `with` и его работа
   - Реализация собственного контекстного менеджера
   - Обработка исключений в `__exit__`
   - Вложенные контекстные менеджеры
   - Полезные примеры: таймер, блокировка, временная смена директории

2. **Настоящий конструктор: `__new__`**
   - Жизненный цикл объекта: `__new__` → `__init__`
   - Создание объекта (аллокация)
   - Паттерн Singleton
   - Наследование от неизменяемых типов (int, tuple)
   - Кэширование объектов (пул объектов)

3. **Финализатор: `__del__`**
   - Когда вызывается `__del__`
   - Проблемы и ограничения
   - Почему не стоит полагаться на `__del__` для освобождения ресурсов
   - Альтернативы: контекстные менеджеры и явные методы

4. **Итоги и сравнение методов**
   - Таблица применения
   - Ключевые моменты для запоминания
   - Частые ошибки

In [2]:
# Демонстрация проблемы управления ресурсами без контекстного менеджера
f = open('data.txt', 'w')
try:
    f.write('hello')
finally:
    f.close()
print("Файл закрыт с помощью try/finally")

Файл закрыт с помощью try/finally


## 🔐 Менеджеры контекста: `__enter__` и `__exit__`

### Синтаксис `with`

```python
with выражение as переменная:
    блок_кода

In [3]:


# Свой контекстный менеджер для работы с файлом
class ManagedFile:
    def __init__(self, filename, mode='r', encoding='utf-8'):
        self.filename = filename
        self.mode = mode
        self.encoding = encoding
        self.file = None

    def __enter__(self):
        print(f"🟢 Открываем файл {self.filename}")
        self.file = open(self.filename, self.mode, encoding=self.encoding)
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"🔴 Закрываем файл {self.filename}")
        if self.file:
            self.file.close()
        
        if exc_type is not None:
            print(f"⚠️ Исключение: {exc_type.__name__}: {exc_val}")
        return False  # исключение не подавляем

# Использование
print("--- Нормальная работа ---")
with ManagedFile('test.txt', 'w') as f:
    f.write('Привет, мир!')
print("Файл закрыт автоматически\n")

print("--- С исключением ---")
try:
    with ManagedFile('test.txt', 'r') as f:
        content = f.read()
        print(f"Содержимое: {content}")
        raise ValueError("Что-то пошло не так")
except ValueError as e:
    print(f"Поймали исключение: {e}")

--- Нормальная работа ---
🟢 Открываем файл test.txt
🔴 Закрываем файл test.txt
Файл закрыт автоматически

--- С исключением ---
🟢 Открываем файл test.txt
Содержимое: Привет, мир!
🔴 Закрываем файл test.txt
⚠️ Исключение: ValueError: Что-то пошло не так
Поймали исключение: Что-то пошло не так


In [5]:
# Пример: подавление исключений в __exit__
class SuppressError:
    def __init__(self, *exceptions):
        self.exceptions = exceptions

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None and issubclass(exc_type, self.exceptions):
            print(f"Подавляем исключение {exc_type.__name__}: {exc_val}")
            return True  # исключение подавлено
        return False

# print("--- Подавление ValueError ---")
# with SuppressError(ValueError):
#     print("До ошибки")
#     x = int("не число")
#     print("После ошибки (не выполнится)")
# print("Программа продолжается\n")

print("--- ValueError не подавляется, если не указан ---")
try:
    with SuppressError(TypeError):
        x = int("не число")
except ValueError:
    print("ValueError пробросилось наружу")

--- ValueError не подавляется, если не указан ---
ValueError пробросилось наружу


In [6]:
# Вложенные контекстные менеджеры
with ManagedFile('out1.txt', 'w') as f1, ManagedFile('out2.txt', 'w') as f2:
    f1.write('Первый файл')
    f2.write('Второй файл')
print("Оба файла записаны и закрыты")

🟢 Открываем файл out1.txt
🟢 Открываем файл out2.txt
🔴 Закрываем файл out2.txt
🔴 Закрываем файл out1.txt
Оба файла записаны и закрыты


In [ ]:
# Полезный пример: таймер
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.end = time.perf_counter()
        self.duration = self.end - self.start
        print(f"⏱️ Время выполнения: {self.duration:.6f} сек")

    


with Timer() as t:
    sum(i**2 for i in range(10**7))

print(type(t))

⏱️ Время выполнения: 0.659932 сек
<class '__main__.Timer'>


In [ ]:
# Пример: блокировка для потоков
import threading

class LockContext:
    def __init__(self, lock):
        self.lock = lock

    def __enter__(self):
        self.lock.acquire()
        return self


    def __exit__(self, *args):
        self.lock.release()

lock = threading.Lock()
with LockContext(lock):
    print("Критическая секция")

Критическая секция


In [11]:
# Пример: временная смена директории
import os

class cd:
    def __init__(self, new_path):
        self.new_path = new_path
        self.saved_path = None

    def __enter__(self):
        self.saved_path = os.getcwd()
        os.chdir(self.new_path)
        return self

    def __exit__(self, *args):
        os.chdir(self.saved_path)

# Создадим временную папку для демонстрации
import tempfile
tmpdir = tempfile.mkdtemp()
print(f"Создана временная папка: {tmpdir}")

with cd(tmpdir):
    print(f"Текущая директория: {os.getcwd()}")
    with open('test.txt', 'w') as f:
        f.write('test')
print(f"Вернулись в исходную: {os.getcwd()}")

#Удалим временную папку
import shutil
shutil.rmtree(tmpdir)

Создана временная папка: /var/folders/bc/xst6vkw90bg1l4_bdw16gtdc0000gn/T/tmpr1046o5g
Текущая директория: /private/var/folders/bc/xst6vkw90bg1l4_bdw16gtdc0000gn/T/tmpr1046o5g
Вернулись в исходную: /Users/max/Documents/PythonProjects/Python_OOP/lectures


In [ ]:
# Пример: временная смена директории
import os

class cd:
    def __init__(self, new_path):
        self.new_path = new_path
        self.saved_path = None

    def __enter__(self):
        self.saved_path = os.getcwd()
        os.chdir(self.new_path)
        return self

    def __exit__(self, *args):
        os.chdir(self.saved_path)

# Создадим временную папку для демонстрации
import tempfile
tmpdir = tempfile.mkdtemp()
print(f"Создана временная папка: {tmpdir}")

with cd(tmpdir):
    print(f"Текущая директория: {os.getcwd()}")
    with open('test.txt', 'w') as f:
        f.write('test')
print(f"Вернулись в исходную: {os.getcwd()}")

#Удалим временную папку
import shutil
shutil.rmtree(tmpdir)

Создана временная папка: /var/folders/bc/xst6vkw90bg1l4_bdw16gtdc0000gn/T/tmpr1046o5g
Текущая директория: /private/var/folders/bc/xst6vkw90bg1l4_bdw16gtdc0000gn/T/tmpr1046o5g
Вернулись в исходную: /Users/max/Documents/PythonProjects/Python_OOP/lectures


## 🏗️ Настоящий конструктор: `__new__`

### Жизненный цикл объекта: `__new__` → `__init__`

- `__new__` — создаёт объект (конструктор)
- `__init__` — инициализирует объект (инициализатор)

In [12]:
class Demo:
    def __new__(cls, *args, **kwargs):
        print("1. __new__ called")
        instance = super().__new__(cls)
        print("2. Объект создан")
        return instance

    def __init__(self, value):
        print("3. __init__ called")
        self.value = value

d = Demo(42)
print(f"Объект: {d}, значение: {d.value}")

1. __new__ called
2. Объект создан
3. __init__ called
Объект: <__main__.Demo object at 0x10d3002f0>, значение: 42


In [ ]:
# Пример 1: Singleton (одиночка)
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            print("Создаём единственный экземпляр")
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value=None):
        if not hasattr(self, 'initialized'): #setattr, getattr, delattr
            self.value = value
            self.initialized = True
            print(f"Инициализация со значением {value}")
        else:
            print(f"Повторный вызов __init__, значение остаётся {self.value}")

a = Singleton(10)
b = Singleton(20)
print(f"a is b: {a is b}")
print(f"a.value = {a.value}, b.value = {b.value}")

Создаём единственный экземпляр
Инициализация со значением 10
Повторный вызов __init__, значение остаётся 10
a is b: True
a.value = 10, b.value = 10


In [14]:
# Пример 2: Наследование от неизменяемого типа (int)
class PositiveInteger(int):
    def __new__(cls, value):
        if value < 0:
            value = -value
        return super().__new__(cls, value)

p = PositiveInteger(-5)
print(p, type(p))

5 <class '__main__.PositiveInteger'>


In [16]:
# Пример 3: Кэширование объектов (пул)
class Color:
    _cache = {}

    def __new__(cls, name, hex_code):
        if name in cls._cache:
            print(f"Возвращаем из кэша: {name}")
            return cls._cache[name]
        print(f"Создаём новый цвет: {name}")
        instance = super().__new__(cls)
        instance.name = name
        instance.hex_code = hex_code
        cls._cache[name] = instance
        return instance

red1 = Color('red', '#FF0000')
red2 = Color('red', '#FF0000')
blue = Color('blue', '#0000FF')
print(red1 is red2)
print(Color._cache)

Создаём новый цвет: red
Возвращаем из кэша: red
Создаём новый цвет: blue
True
{'red': <__main__.Color object at 0x10d300830>, 'blue': <__main__.Color object at 0x10d102ad0>}


In [23]:
# Пример 4: Подкласс tuple (неизменяемый)
class Point(tuple):
    def __new__(cls, x, y):
        return super().__new__(cls, (x, y))
    
    @property
    def x(self):
        return self[0]

    @property
    def y(self):
        return self[1]

p = Point(3, 4)
# print(p
print(p)

(3, 4)


## 🗑️ Финализатор: `__del__`

### Когда вызывается `__del__`?
Метод `__del__` вызывается сборщиком мусора, когда объект становится недоступным (количество ссылок на него равно нулю). В CPython это происходит сразу, но из-за циклических ссылок может задерживаться.

In [ ]:
class DemoDel:
    def __init__(self, name):
        self.name = name
        print(f"Создан {self.name}")

    def __del__(self):
        print(f"Уничтожен {self.name}")

a = DemoDel("A")
b = DemoDel("B")
del a   # сразу вызовет __del__
print("Конец программы")

Создан A
Создан B
Уничтожен B
Конец программы


In [26]:
# Проблема: циклические ссылки и __del__
import gc

class Parent:
    def __init__(self, child):
        self.child = child
    def __del__(self):
        print("Parent deleted")

class Child:
    def __init__(self, parent):
        self.parent = parent
    def __del__(self):
        print("Child deleted")

# Создаём цикл
p = Parent(None)
c = Child(p)
p.child = c

# Удаляем внешние ссылки
del p
del c

# Принудительная сборка мусора не удалит объекты с __del__
gc.collect()
print("Цикл не удалён")

Parent deleted
Child deleted
Цикл не удалён


### Почему не стоит использовать `__del__` для освобождения ресурсов?

- Не гарантирует своевременного вызова.
- Может не вызваться вовсе (при завершении программы).
- Усложняет работу сборщика мусора.
- Исключения в `__del__` игнорируются.

**Альтернатива:** контекстные менеджеры.

In [ ]:
# Правильный подход: контекстный менеджер
class GoodResource:
    def __init__(self, name):
        self.name = name
        self._opened = True
        print(f"Открыт ресурс {name}")

    def close(self):
        if self._opened:
            print(f"Закрыт ресурс {self.name}")
            self._opened = False

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

with GoodResource("база данных") as res:
    print("Работаем с ресурсом")
# автоматически закрыт

## 📊 Сравнение методов

| Метод          | Назначение                               | Особенности                                                                 |
|----------------|------------------------------------------|-----------------------------------------------------------------------------|
| `__enter__`    | Вход в контекстный менеджер              | Возвращает объект для `as`. Вызывается при входе в `with`.                 |
| `__exit__`     | Выход из контекстного менеджера          | Получает информацию об исключении. Возврат `True` подавляет исключение.    |
| `__new__`      | Создание объекта (конструктор)           | Статический метод. Возвращает новый экземпляр.                             |
| `__init__`     | Инициализация объекта                    | Вызывается после `__new__`. Заполняет атрибуты.                            |
| `__del__`      | Уничтожение объекта (финализатор)        | Вызывается сборщиком мусора. Ненадёжен.                                    |

## 🎯 Ключевые моменты для запоминания

1. **Менеджеры контекста** (`with`) — лучший способ гарантировать освобождение ресурсов. Реализуйте `__enter__` и `__exit__` в своих классах для безопасной работы.

2. **`__new__`** — вызывается до `__init__`. Используйте его для:
   - Создания синглтонов
   - Наследования от неизменяемых типов
   - Кэширования объектов

3. **`__del__`** — ненадёжен. Не полагайтесь на него для критических ресурсов. Используйте контекстные менеджеры или явные методы.

## ❌ Частые ошибки

- Забыть вызвать `super().__new__(cls)` в `__new__`.
- Использовать `__del__` для закрытия файлов или соединений.
- В `__exit__` не обрабатывать параметры исключения, но при этом подавлять все ошибки (возвращая `True`).
- Пытаться изменять неизменяемый объект в `__init__` (например, кортеж) — нужно делать это в `__new__`.

